# 2장 — 이어쓰기 기계의 한계 🧱

웹 챕터의 핵심 문장은 이것이었죠. 베이스 모델의 전부는 **P(다음 단어 | 앞 문맥)**.

이번 노트북에서는 그 **조건부 분포를 직접 눈으로** 봅니다.
문맥이 바뀌면 분포가 어떻게 밀리는지(퓨샷의 원리), 그리고 예시 밖 문맥에서는
어떻게 이어쓰기 본능으로 후퇴하는지(퓨샷의 한계)까지.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch2')

## 1. 조건부 분포를 눈으로 보기

이어쓰기 기계(워밍업)의 빈도표가 사실은 **조건부 확률분포**였습니다.
"앞 단어가 이것일 때, 다음 단어의 후보와 확률", 그걸 출력하는 함수를 만들어요.

In [ ]:
from collections import Counter, defaultdict

# 이 기계의 '인터넷' — 일상 글과 Q/A 형식 글이 섞여 있다
CORPUS = [
    '오늘 날씨가 좋다',
    '오늘 점심 메뉴 추천',
    '오늘 회의 일정 공유',
    '오늘 본 영화 후기',
    '요청: 존댓말로 바꿔줘 답변: 알겠습니다',
    '요청: 요약해줘 답변: 네',
    '요청: 번역해줘 답변: 물론이죠',
]

next_words = defaultdict(Counter)
for line in CORPUS:
    words = line.split()
    for a, b in zip(words, words[1:]):
        next_words[a][b] += 1

def show_dist(context_word):
    """P(다음 단어 | 앞 단어) — 후보·확률·갈래 수를 출력한다."""
    counter = next_words.get(context_word)
    if not counter:
        print(f"'{context_word}' 뒤: (본 적 없음, 분포가 비어 있다)")
        return 0
    total = sum(counter.values())
    print(f"'{context_word}' 뒤에 올 단어의 분포: 갈래 {len(counter)}개")
    for word, count in counter.most_common():
        print(f'   {word:10s} {count / total:.0%}')
    return len(counter)

branches_broad = show_dist('오늘')
print()
branches_narrow = show_dist('답변:')

**관찰**: '오늘' 뒤는 갈래가 넓고(날씨·점심·회의·영화…), '답변:' 뒤는 좁습니다(공손한 대답들만).

이게 퓨샷의 전부예요. 예시를 붙여 문맥이 `답변:` 으로 끝나게 만들면,
모델은 **좁혀진 분포**(공손한 대답 갈래)에서 다음 단어를 고르게 됩니다.
5부 7장의 언어로: **문맥이 퍼플렉시티(실질 갈래 수)를 줄인 것**.

In [ ]:
# 갈래 수 비교 — 문맥이 분포를 얼마나 좁혔나
print(f"'오늘'   뒤 갈래: {branches_broad}개  (일상 글: 넓다)")
print(f"'답변:'  뒤 갈래: {branches_narrow}개  (Q/A 형식 문맥: 좁다)")

assert branches_narrow < branches_broad, '형식 문맥이 갈래를 줄여야 한다!'
print('\n✔ 확인: 문맥이 다음 단어의 갈래를 줄인다 (퓨샷이 작동하는 이유)')

## 2. 한계 재현: 예시 밖 문맥에서는?

퓨샷 예시는 `요청: … 답변: …` 형식만 밀어 줬습니다.
그럼 예시에 없는 형식(예를 들어 `명령:`)으로 말을 걸면 어떻게 될까요?

In [ ]:
# 예시 밖 문맥 — 분포가 비어 있다
branches_unknown = show_dist('명령:')

# 분포가 비면 기계는 어떻게 할까? 진짜 모델은 '가장 그럴듯한 일반 글'로 후퇴한다.
# 우리 초소형 기계로는 전체 코퍼스에서 가장 흔한 시작 단어로 이어쓰는 것으로 흉내 낸다.
all_starts = Counter(line.split()[0] for line in CORPUS)
fallback = all_starts.most_common(1)[0][0]
print(f"\n'명령: 보고서 써' 에 대한 반응 → 형식을 모르니 평소 하던 이어쓰기로 후퇴: '{fallback} …'")

assert branches_unknown == 0, '예시 밖 문맥은 분포가 비어 있어야 한다'
print('\n✔ 확인: 예시가 못 미는 곳에서는 이어쓰기 본능으로 돌아간다 (퓨샷의 한계 ②)')

## 3. 관찰 정리 + 🤖 AI 연결

- **P(다음 단어 | 앞 문맥)**: 조건(문맥)이 바뀌면 분포가 바뀐다. 퓨샷 = 조건 조작.
- 형식 문맥(`답변:`)은 갈래를 좁히고(퍼플렉시티↓, 5부 7장), 예시 밖 문맥에서는 분포가 비어
  이어쓰기 본능으로 후퇴한다. 웹 챕터 돌발 실험의 코드 버전.
- 그리고 무엇을 해도 `next_words` 표(= 가중치)는 **한 칸도 바뀌지 않았다**는 점.
  표 자체를 바꾸는 것, 그게 다음 장의 SFT입니다.

**다음 챕터**: 3장(SFT). 모범 답안으로 가중치를 진짜 바꿉니다. 손실은 5부의 크로스 엔트로피 그대로.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('ch2')`.